# Data Augmentation per la sicurezza delle centrali elettriche

CyberEye Solutions, un leader emergente nel settore della sicurezza cibernetica per infrastrutture critiche, si trova di fronte a una crescente sfida nella protezione delle centrali elettriche contro minacce informatiche avanzate. Attualmente, il sistema di sorveglianza delle centrali utilizza tecnologie di riconoscimento di immagini per identificare e reagire tempestivamente a situazioni potenzialmente pericolose. Tuttavia, la capacità di riconoscere con precisione e tempestività oggetti e comportamenti critici nelle immagini è limitata dai dataset di addestramento attualmente disponibili, che non rappresentano appieno la variabilità e la complessità delle situazioni reali.

Attualmente, i modelli esistenti basati su dataset limitati non riescono a rilevare con la necessaria precisione anomalie o minacce potenziali nelle immagini, compromettendo la capacità di risposta e mitigazione dell'azienda di fronte a situazioni di emergenza. Migliorare la capacità di identificare tempestivamente oggetti e comportamenti critici nelle immagini è fondamentale per garantire la continuità operativa e la sicurezza delle infrastrutture critiche gestite da CyberEye Solutions.

## Benefici della Soluzione

**Miglioramento della Sicurezza delle Infrastrutture Critiche:** Espandere il dataset utilizzando tecniche avanzate di Data Augmentation consentirà di migliorare l'accuratezza del sistema di riconoscimento di immagini. Un modello più preciso e affidabile sarà in grado di rilevare con maggiore tempestività e precisione comportamenti sospetti o minacce potenziali nelle immagini delle centrali elettriche, migliorando così la sicurezza delle infrastrutture critiche e riducendo il rischio di incidenti o sabotaggi.

**Efficienza Operativa e Riduzione del Tempo di Risposta:** Automatizzando il processo di generazione di nuovi dati attraverso la creazione di immagini e testi variati, CyberEye Solutions ottimizzerà l'efficienza operativa. Questo permetterà all'azienda di concentrare le risorse umane su attività di analisi e mitigazione delle minacce, riducendo il tempo di risposta agli eventi critici e migliorando la capacità di gestione delle emergenze.

**Innovazione Tecnologica nel Settore della Sicurezza:** Utilizzando tecniche avanzate di deep learning e generazione di dati, CyberEye Solutions promuoverà l'innovazione nel campo della sicurezza cibernetica per infrastrutture critiche. L'implementazione di modelli di riconoscimento di immagini più sofisticati non solo migliorerà la sicurezza delle centrali elettriche, ma dimostrerà anche l'impegno dell'azienda nell'adozione di tecnologie all'avanguardia per affrontare le sfide emergenti nel settore della sicurezza cibernetica.

## Dettagli del Progetto

- **Acquisizione del Dataset:** Utilizzare il dataset OxfordIIITPet da PyTorch come base per il progetto di miglioramento del sistema di riconoscimento di immagini per infrastrutture critiche.
- **Image Captioning e Generazione di Dati:** Applicare l’image captioning per creare descrizioni iniziali delle immagini. 
Successivamente, utilizzare un modello generativo di testo per produrre varianti o descrizioni analoghe. 
Infine, impiegare un modello generativo di immagini per creare nuove immagini a partire dalle caption originali o dai testi generati, arricchendo così il dataset con dati sintetici.
- **Addestramento del Modello:** Addestrare un modello di riconoscimento di immagini utilizzando il dataset esteso, valutando la qualità dei dati prodotti e confrontando le performance del modello su dataset ridotto e dataset incrementato.
- **Valutazione delle Performance:** Misurare l'accuracy, precision, recall e altre metriche di performance per confrontare il modello addestrato su entrambi i dataset. Commentare le differenze nelle performance e l'efficacia delle tecniche di Data Augmentation nel migliorare l'accuratezza del modello in contesti reali di sicurezza delle infrastrutture critiche.

## Conclusioni

CyberEye Solutions si impegna a rafforzare la sicurezza delle infrastrutture critiche attraverso l'implementazione di soluzioni avanzate di riconoscimento di immagini. Utilizzando approcci innovativi e tecnologie all'avanguardia, l'azienda mira non solo a migliorare l'efficacia dei suoi sistemi di sicurezza cibernetica, ma anche a definire nuovi standard nel settore per la protezione delle infrastrutture critiche contro le minacce informatiche sempre più sofisticate.

# Implementazione

## Import delle dipendenze

Il progetto implementa un'analisi completa per **CyberEye Inc**, al fine di classificare automaticamente e correttamente le immagini di animali, per la sicurezza delle infrastrutture. L'analisi sarà completata dall'aggiunta della **Generative AI** tramite modelli sviluppati ad-hoc e pretrained per la data augmentation.

Utilizzeremo, per rendere il codice più modulare e alleggerire questo notebook, un modulo custom chiamato **DNNHelper**, che contiene al suo interno importanti classi e metodi per l'agevolazione di training, testing, cross-validation e plotting dei risultati.

Effettuiamone quindi il download e la registrazione:

In [ ]:
!git clone https://github.com/crypto-infinity/dnnhelper
%pip install dnnhelper/
%cd dnnhelper

Cominciamo quindi con l'analisi delle dipendenze necessarie e il relativo import nel notebook:

In [ ]:
#Dependencies import

#Generics
import os

#PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer, TransformerDecoder, TransformerDecoderLayer

#Albumentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

#Dataset
from torchvision.datasets import OxfordIIITPet

#PIL
from PIL import Image

#Utility
import re
from tqdm import tqdm
import numpy as np
from matplotlib import pyplot as plt

#HuggingFace Utility
from transformers import AutoTokenizer
from transformers import BlipProcessor, BlipForConditionalGeneration #Image Captioning

#DNNHelper
from dnnhelper import Helper

In [ ]:
%cd ..

Definiamo, al termine del nostro setup, alcune costanti utili e alcune impostazioni del framework Deep Learning che useremo, PyTorch, come il device per il training dei modelli e il random seed per la riproducibilità:

In [ ]:
ITERATIONS = [0, 16, 76, 99]

In [ ]:
DEVICE = Helper.set_device()
print(f"Using device: {DEVICE}")

In [ ]:
#Random seed for reproducibility

SEED = 56

Helper.set_seed(SEED)
print(f"Random seed set to: {SEED}")

Definiamo i modelli pre-trained che utilizzeremo:

In [ ]:
CAPTIONER = "Salesforce/blip-image-captioning-base"
DESCRIPTIONS = "gpt2"
DIFFUSER = ""

Infine, creiamo il wrapper per la liberia di data augmentation, Albumentations:

In [ ]:
# Wrapper for Albumentations transforms

class Transforms:
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, img, *args, **kwargs):
        return self.transforms(image=np.array(img))['image']

## Acquisizione e analisi del dataset

Cominciamo con l'importazione del dataset (iniziamo da quello di train per una prima analisi, dato che il file zip è già diviso in train e test, che ci sarà molto utile):

In [ ]:
#For debug purposes only

import ssl
ssl._create_default_https_context = ssl._create_unverified_context

E definiamo una prima pipeline di pre-processing, ridimensionando le immagini alla shape [3, 256, 256] per agevolare i tempi di training e normalizzandone i valori dei tensori:

In [ ]:
# Preprocessing default template

default_preprocessing_pipeline = A.Compose([
            A.Resize(256, 256),
            A.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
            ToTensorV2(),
        ])

Carichiamo quindi le immagini, e conduciamo una prima analisi esplorativa, visualizzando anche alcuni esempi:

In [ ]:
train_ds = OxfordIIITPet(root="dataset", download=True, split="trainval", transform=Transforms(default_preprocessing_pipeline))
test_ds = OxfordIIITPet(root="dataset", download=False, split="test", transform=Transforms(default_preprocessing_pipeline))

In [ ]:
for i in ITERATIONS:    
    Helper.plot_images(train_ds, train_ds.classes, i)

# Data Augmentation: tecniche e pipeline

## Image Captioning: generazione descrizioni immagini

In [ ]:
def caption_images(dataset, captioner, processor, save_to_file=True):
    captions = []

    for i in range(len(dataset)):
        raw_image = Image.open(dataset._images[i])
        inputs = processor(raw_image, return_tensors="pt").to(DEVICE)
        output = captioner.generate(**inputs)
        captions.append(processor.decode(output[0], skip_special_tokens=True))

    if save_to_file:
        os.makedirs("captioning", exist_ok=True)
        with open("captioning/captions.txt", "w") as caption_f:
            caption_f.writelines(captions)
    
    return captions

In [ ]:
captioner = BlipForConditionalGeneration.from_pretrained(CAPTIONER).to(DEVICE)
captioner_proc = BlipProcessor.from_pretrained(CAPTIONER)

In [ ]:
captions = caption_images(train_ds, captioner, captioner_proc, True)

## Generazione di varianti testuali (NLP)

## Generazione di nuove immagini da caption

## Addestramento e valutazione del modello

## Analisi dei risultati e confronto

## Conclusioni e sviluppi futuri